# FCOS（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出における代表的なAnchor-free（アンカーフリー）手法であるFCOS（Fully Convolutional One-Stage Object Detection）を，PASCAL VOC 2007データセットを用いて実装する．SSD・RetinaNet・EfficientDetなど，これまでの1段階検出器はいずれも，特徴マップの各セルにあらかじめ複数サイズ・アスペクト比のAnchor（デフォルトボックス）を格子状に配置し，IoUに基づいてGTボックスとマッチングさせる設計であった．FCOSは，このAnchorを一切使わず，特徴ピラミッドの**各位置（ピクセル）が直接，クラス・境界までの距離・中心度（centerness）を予測する**という全く異なるアプローチを取る．Anchorの形状・スケールという手動設計のハイパーパラメータを排除しつつ，SSD/RetinaNetに匹敵する精度を達成できることがFCOSの核心である．

## モジュールのインポートとGPUの確認
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import math
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込む．データセットの詳細（クラス構成やダウンロード方法）は`ssd.ipynb`と同じであるため，ここでは繰り返さない．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してほしい．

FCOSはFPNにより複数のストライド（8, 16, 32, 64, 128）の特徴マップを使用するため，入力画像サイズは最大ストライドである128の倍数に設定する必要がある．`retinanet.ipynb`と同様に`IMG_SIZE = 512`とする．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES) + 1  # 背景を含めたクラス数（分類ヘッドの設計は後述）
IMG_SIZE = 512

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景含む）:', n_class)

## FCOS：バックボーンとFeature Pyramid Network（FPN）
FCOSも`retinanet.ipynb`・`fpn.ipynb`と同様に，ResNet50バックボーンの上にFPNを組み合わせ，どのレベルでも意味的に強い特徴を持つ特徴ピラミッドを構築する．ResNet50の`layer2`・`layer3`・`layer4`の出力（C3, C4, C5，ストライド8, 16, 32）にトップダウンパス＋横方向結合（lateral connection）を適用してP3, P4, P5を作る構成は`retinanet.ipynb`の`FPNNeck`と全く同じである．FCOSの原論文では大きな物体に対応するため，さらに2段階深いP6（ストライド64）・P7（ストライド128）を加えた5段階のピラミッドを標準的に使用するため，本ノートブックでもP3〜P7の5レベルとする（`retinanet.ipynb`はP3〜P6の4レベル）．P6はC5に3×3ストライド2畳み込みを適用して作り，P7はP6にReLUを適用したのちさらに3×3ストライド2畳み込みを適用して作る．

In [ ]:
class FPNNeck(nn.Module):
    """ResNetのC3, C4, C5から，トップダウンパス＋横方向結合でP3〜P7を作るFPN
    （retinanet.ipynbのFPNNeckに，大きな物体向けのP7を1段追加したもの）"""
    def __init__(self, in_channels_list=(512, 1024, 2048), out_channels=256):
        super().__init__()
        c3_ch, c4_ch, c5_ch = in_channels_list
        self.lateral3 = nn.Conv2d(c3_ch, out_channels, kernel_size=1)
        self.lateral4 = nn.Conv2d(c4_ch, out_channels, kernel_size=1)
        self.lateral5 = nn.Conv2d(c5_ch, out_channels, kernel_size=1)

        self.smooth3 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.p6 = nn.Conv2d(c5_ch, out_channels, kernel_size=3, stride=2, padding=1)  # C5からストライド2でP6（ストライド64）
        self.p7 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=2, padding=1)  # P6からさらにストライド2でP7（ストライド128）

    def forward(self, c3, c4, c5):
        p5 = self.lateral5(c5)
        p4 = self.lateral4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode='nearest')
        p3 = self.lateral3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode='nearest')

        p3 = self.smooth3(p3)
        p4 = self.smooth4(p4)
        p5 = self.smooth5(p5)
        p6 = self.p6(c5)
        p7 = self.p7(F.relu(p6))

        return [p3, p4, p5, p6, p7]  # ストライド 8, 16, 32, 64, 128


class FCOSBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool, resnet.layer1)
        self.layer2 = resnet.layer2  # C3, stride  8, channels  512
        self.layer3 = resnet.layer3  # C4, stride 16, channels 1024
        self.layer4 = resnet.layer4  # C5, stride 32, channels 2048
        self.fpn = FPNNeck(in_channels_list=(512, 1024, 2048), out_channels=256)

    def forward(self, x):
        x = self.stem(x)
        c3 = self.layer2(x)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        return self.fpn(c3, c4, c5)  # [P3, P4, P5, P6, P7]


FPN_STRIDES = [8, 16, 32, 64, 128]
FPN_OUT_CHANNELS = 256

### 特徴ピラミッドの各位置とレベルごとの守備範囲
Anchorベースの手法では，特徴マップの各セルに複数サイズのAnchorを配置することで，様々な大きさの物体に対応していた．FCOSはAnchorを持たない代わりに，各セル（location）をそのまま入力画像上の1点とみなし，ストライド$s$のレベルでは画像上の座標$(s/2 + j \cdot s,\ s/2 + i \cdot s)$（$i, j$はセルの行・列インデックス）にその点が対応すると考える．

そして，Anchorのスケールの代わりに，各ピラミッドレベルに「回帰対象の大きさ（$\max(l, t, r, b)$）がこの範囲に収まる場合のみ，このレベルが担当する」という**サイズ範囲（size range）**を割り当てる．浅いレベル（P3）は小さい物体，深いレベル（P7）は大きい物体を担当するように，範囲を単調に大きく設定する．

In [ ]:
SIZE_RANGES = [(0, 64), (64, 128), (128, 256), (256, 512), (512, float('inf'))]  # P3〜P7が担当する回帰対象の大きさの範囲


def generate_locations(feature_shapes, strides):
    """各特徴マップの各セルに対応する，入力画像上での座標(x, y)をレベルごとのリストとして計算する"""
    locations = []
    for (h, w), stride in zip(feature_shapes, strides):
        shifts_x = (torch.arange(w, device=device) + 0.5) * stride
        shifts_y = (torch.arange(h, device=device) + 0.5) * stride
        shift_y, shift_x = torch.meshgrid(shifts_y, shifts_x, indexing='ij')
        locations.append(torch.stack([shift_x.reshape(-1), shift_y.reshape(-1)], dim=1))  # (h*w, 2), (x, y)
    return locations


FEATURE_SHAPES = [(IMG_SIZE // s, IMG_SIZE // s) for s in FPN_STRIDES]
LOCATIONS = generate_locations(FEATURE_SHAPES, FPN_STRIDES)  # レベルごとの(num_locs_level, 2)のリスト
print('各レベルの位置数:', [loc.shape[0] for loc in LOCATIONS], ' 合計:', sum(loc.shape[0] for loc in LOCATIONS))

## 検出ヘッド（分類・回帰・centerness）
FCOSの検出ヘッドは，`retinanet.ipynb`の`RetinaNetHead`と同様に**全てのピラミッドレベル（P3〜P7）で重みを共有する**畳み込みサブネットワークである．分類用・回帰用にそれぞれ4層の3×3畳み込み（チャンネル数256）のタワーを持ち，その出力から3種類のヘッドを計算する．

- **分類ヘッド**：`n_class - 1`チャンネル．RetinaNetと同様に，背景クラスを明示的に持たない設計（前景の各クラスを独立した2値分類器として扱う）を取る．「物体なし」は，全チャンネルのスコアが低いことで暗黙的に表現される．学習初期の安定化のため，最終層のバイアスを`retinanet.ipynb`と同じ$-\log((1-\pi)/\pi)$（$\pi=0.01$）で初期化する．
- **回帰ヘッド**：4チャンネル（l, t, r, b）．出力される距離は必ず非負であるため，畳み込みの生出力にレベルごとの学習可能なスケール（`self.scales`）を掛けたのち`exp()`を適用し，正の値に変換する．レベルごとにスケールを学習させることで，各レベルが担当する距離の範囲（表を参照）に自動的に適応できる．
- **centernessヘッド**：1チャンネル．回帰タワーの出力から分岐させる（centernessは回帰と同様，位置が物体の中心にどれだけ近いかという「幾何的な」情報のため）．

In [ ]:
class FCOSHead(nn.Module):
    """全レベルで重みを共有する検出ヘッド（retinanet.ipynbのRetinaNetHeadと同じ設計思想）"""
    def __init__(self, in_channels, n_fg_class, num_levels, prior_prob=0.01):
        super().__init__()
        self.n_fg_class = n_fg_class

        def make_tower():
            layers = []
            for _ in range(4):
                layers += [nn.Conv2d(in_channels, in_channels, 3, padding=1), nn.ReLU(inplace=True)]
            return nn.Sequential(*layers)

        self.cls_tower = make_tower()
        self.reg_tower = make_tower()

        self.cls_out = nn.Conv2d(in_channels, n_fg_class, kernel_size=3, padding=1)
        self.reg_out = nn.Conv2d(in_channels, 4, kernel_size=3, padding=1)
        self.centerness_out = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)

        # 学習初期の安定化のため，分類ヘッドのバイアスを背景寄りに初期化する（retinanet.ipynbと同じ工夫）
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        nn.init.constant_(self.cls_out.bias, bias_value)

        # ピラミッドレベルごとに回帰出力のスケールを調整する学習可能なパラメータ
        self.scales = nn.ParameterList([nn.Parameter(torch.tensor(1.0)) for _ in range(num_levels)])

    def forward(self, features):
        """featuresはP3〜P7のリスト．全レベルで同じ重み（self.cls_towerなど）を使い回す"""
        cls_outputs, reg_outputs, centerness_outputs = [], [], []
        for level, feat in enumerate(features):
            cls_feat = self.cls_tower(feat)
            reg_feat = self.reg_tower(feat)

            cls_out = self.cls_out(cls_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.n_fg_class)
            reg_raw = self.reg_out(reg_feat)
            reg_out = torch.exp(self.scales[level] * reg_raw).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4)
            centerness_out = self.centerness_out(reg_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1)

            cls_outputs.append(cls_out)
            reg_outputs.append(reg_out)
            centerness_outputs.append(centerness_out)

        return cls_outputs, reg_outputs, centerness_outputs  # レベルごとのリストのまま返す（レベルごとに位置数が異なるため）


class FCOS(nn.Module):
    def __init__(self, n_class):
        super().__init__()
        self.backbone = FCOSBackbone()
        self.head = FCOSHead(FPN_OUT_CHANNELS, n_fg_class=n_class - 1, num_levels=len(FPN_STRIDES))

    def forward(self, x):
        features = self.backbone(x)  # [P3, P4, P5, P6, P7]
        return self.head(features)

## 各位置への教師信号の割り当て（アンカーフリー）
これがFCOSとAnchorベースの手法（SSD・RetinaNet）との最大の違いである．SSD・RetinaNetでは，「IoUが閾値以上となるAnchor」をGTボックスに割り当てるマッチング処理（`match_default_boxes`・`match_anchors`）が必要だった．FCOSは，Anchorという中間表現を持たず特徴ピラミッドの各位置を直接使うため，IoUマッチングの代わりに次の2条件だけで正例（positive）かどうかを判定する．

1. その位置が，GTボックスの内部にある（$l, t, r, b$がすべて正）．
2. その位置での回帰対象の大きさ$\max(l, t, r, b)$が，そのレベルに割り当てられたサイズ範囲（`SIZE_RANGES`）に収まる．

両方を満たす位置だけが，そのGTボックスの正例になる．1つの位置が複数のGTボックスの条件を同時に満たしてしまった場合（物体同士が重なっている場合など）は，**面積が最も小さいGTボックスを優先して割り当てる**（標準的なタイブレークのルール．小さい物体を優先することで，大きい物体に埋もれて見逃されるのを防ぐ）．どちらの条件も満たさない位置は，どのクラスにも属さない負例として扱う．正例の位置では，割り当てられたGTボックスから$(l, t, r, b)$と，centernessの教師信号$\sqrt{\frac{\min(l,r)}{\max(l,r)} \times \frac{\min(t,b)}{\max(t,b)}}$を計算する．

In [ ]:
def assign_targets(gt_boxes, gt_labels, locations, size_ranges):
    """1枚の画像について，全FPNレベルを通した各位置(location)に対する分類・回帰・centernessの教師信号を作成する"""
    all_locations = torch.cat(locations, dim=0)  # (total_locs, 2)
    level_ranges = torch.cat([
        all_locations.new_tensor(r).expand(loc.size(0), 2) for loc, r in zip(locations, size_ranges)
    ], dim=0)  # (total_locs, 2)

    num_locs = all_locations.size(0)
    cls_target = torch.full((num_locs,), -1, dtype=torch.long, device=all_locations.device)  # -1: 負例（背景）
    reg_target = torch.zeros(num_locs, 4, device=all_locations.device)
    centerness_target = torch.zeros(num_locs, device=all_locations.device)

    if gt_boxes.shape[0] == 0:
        pos_mask = torch.zeros(num_locs, dtype=torch.bool, device=all_locations.device)
        return cls_target, reg_target, centerness_target, pos_mask

    xs, ys = all_locations[:, 0], all_locations[:, 1]
    l = xs[:, None] - gt_boxes[None, :, 0]
    t = ys[:, None] - gt_boxes[None, :, 1]
    r = gt_boxes[None, :, 2] - xs[:, None]
    b = gt_boxes[None, :, 3] - ys[:, None]
    ltrb = torch.stack([l, t, r, b], dim=2)  # (num_locs, num_gt, 4)

    inside_box = ltrb.min(dim=2).values > 0  # 各位置がGTボックスの内部にあるか
    max_ltrb = ltrb.max(dim=2).values  # (num_locs, num_gt)
    inside_range = (max_ltrb >= level_ranges[:, 0:1]) & (max_ltrb <= level_ranges[:, 1:2])
    candidate = inside_box & inside_range  # (num_locs, num_gt)

    # 複数のGTボックスに該当する位置は，面積が最小のGTボックスを優先する（候補にならないGTは無限大にして除外）
    gt_area = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])
    area_expand = gt_area[None, :].expand(num_locs, -1).clone()
    area_expand[~candidate] = float('inf')

    min_area, min_idx = area_expand.min(dim=1)
    pos_mask = min_area < float('inf')

    pos_idx = torch.nonzero(pos_mask, as_tuple=True)[0]
    matched_gt_idx = min_idx[pos_idx]

    cls_target[pos_idx] = gt_labels[matched_gt_idx] - 1  # ラベル1〜20 -> 0〜19（背景を除いたインデックス）
    reg_target[pos_idx] = ltrb[pos_idx, matched_gt_idx]

    l_pos, t_pos, r_pos, b_pos = reg_target[pos_idx, 0], reg_target[pos_idx, 1], reg_target[pos_idx, 2], reg_target[pos_idx, 3]
    centerness_target[pos_idx] = torch.sqrt(
        (torch.min(l_pos, r_pos) / torch.max(l_pos, r_pos).clamp(min=1e-8)) *
        (torch.min(t_pos, b_pos) / torch.max(t_pos, b_pos).clamp(min=1e-8))
    )

    return cls_target, reg_target, centerness_target, pos_mask

## 損失関数：Focal Loss + IoU Loss + centerness BCE
FCOSの損失は3つの要素の和で構成される．

- **分類損失（Focal Loss）**：正例・負例を問わず全ての位置で計算する．物体検出では背景（負例）の位置が圧倒的に多く，通常のクロスエントロピーでは学習が背景クラスに支配されてしまうため，`retinanet.ipynb`と同じFocal Loss $FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$（$\gamma=2.0$, $\alpha=0.25$）を用いる．Focal Lossの詳しい説明・数式の解釈は`retinanet.ipynb`を参照してほしい．
- **回帰損失（IoU Loss）**：正例の位置のみで計算する．予測した$(l, t, r, b)$とターゲットの$(l, t, r, b)$は，どちらも同じ位置を基準にしたボックスを表すため，2つのボックスのIoUを直接計算し，$-\log(\mathrm{IoU})$を損失とする（`box_iou`のようなペアワイズ行列ではなく，位置ごとに1対1で対応する要素ごとのIoU計算である点に注意）．4つの距離を独立に回帰するSmooth L1損失と異なり，ボックス全体の重なり具合を直接最適化できる．
- **centerness損失（BCE）**：正例の位置のみで計算する．ターゲットは前セルで説明した$(l, t, r, b)$から計算した値，予測はcenternessヘッドの生出力（ロジット）で，二値交差エントロピーにより学習する．

In [ ]:
def sigmoid_focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    """マルチラベルのFocal Loss（retinanet.ipynbのFocalLossと同じ定式化）．targetsは0/1のone-hot"""
    prob = torch.sigmoid(logits)
    ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p_t = prob * targets + (1 - prob) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t) ** gamma * ce_loss
    return loss.sum()


def iou_loss(pred_ltrb, target_ltrb):
    """予測・ターゲットの(l, t, r, b)から，位置ごと(要素ごと)にIoUを計算しIoU損失(-log(IoU))を返す"""
    pred_l, pred_t, pred_r, pred_b = pred_ltrb.unbind(dim=1)
    target_l, target_t, target_r, target_b = target_ltrb.unbind(dim=1)

    pred_area = (pred_l + pred_r) * (pred_t + pred_b)
    target_area = (target_l + target_r) * (target_t + target_b)

    inter_w = (torch.min(pred_l, target_l) + torch.min(pred_r, target_r)).clamp(min=0)
    inter_h = (torch.min(pred_t, target_t) + torch.min(pred_b, target_b)).clamp(min=0)
    inter_area = inter_w * inter_h

    union_area = pred_area + target_area - inter_area
    iou = inter_area / union_area.clamp(min=1e-8)

    return -torch.log(iou.clamp(min=1e-8))


class FCOSLoss(nn.Module):
    def __init__(self, n_class, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, cls_preds, reg_preds, centerness_preds, cls_targets, reg_targets, centerness_targets, pos_mask):
        num_pos = pos_mask.sum().clamp(min=1)

        # 分類損失: 全ての位置で計算する。正例の位置だけone-hotベクトルを立てる（負例は全チャンネル0のまま）
        cls_target_onehot = torch.zeros_like(cls_preds)
        pos_idx = torch.nonzero(pos_mask, as_tuple=True)
        cls_target_onehot[pos_idx[0], pos_idx[1], cls_targets[pos_idx]] = 1.0
        cls_loss = sigmoid_focal_loss(cls_preds, cls_target_onehot, self.alpha, self.gamma) / num_pos

        # 回帰損失・centerness損失: 正例の位置のみで計算する
        reg_loss = iou_loss(reg_preds[pos_idx], reg_targets[pos_idx]).sum() / num_pos
        centerness_loss = F.binary_cross_entropy_with_logits(
            centerness_preds[pos_idx], centerness_targets[pos_idx], reduction='sum') / num_pos

        return cls_loss, reg_loss, centerness_loss

## ネットワークの作成
バックボーン＋FPN（`FCOSBackbone`），検出ヘッド（`FCOSHead`）を組み合わせて`FCOS`モデルを構築する．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法には他の検出ノートブックと同様にモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用いる．

In [ ]:
model = FCOS(n_class=n_class)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = FCOSLoss(n_class=n_class)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行う．ミニバッチサイズを8，学習エポック数を20とする．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながら`assign_targets`で各位置への教師信号を計算する．分類・回帰・centernessの3つの損失を個別に記録し，その和を最適化する．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_cls, sum_reg, sum_center = 0.0, 0.0, 0.0

    for images, targets in train_loader:
        images = images.to(device)

        cls_outputs, reg_outputs, centerness_outputs = model(images)
        cls_preds = torch.cat(cls_outputs, dim=1)
        reg_preds = torch.cat(reg_outputs, dim=1)
        centerness_preds = torch.cat(centerness_outputs, dim=1)

        batch_cls_t, batch_reg_t, batch_center_t, batch_pos = [], [], [], []
        for t in targets:
            cls_t, reg_t, center_t, pos = assign_targets(t['boxes'].to(device), t['labels'].to(device), LOCATIONS, SIZE_RANGES)
            batch_cls_t.append(cls_t)
            batch_reg_t.append(reg_t)
            batch_center_t.append(center_t)
            batch_pos.append(pos)
        batch_cls_t = torch.stack(batch_cls_t)
        batch_reg_t = torch.stack(batch_reg_t)
        batch_center_t = torch.stack(batch_center_t)
        batch_pos = torch.stack(batch_pos)

        cls_loss, reg_loss, centerness_loss = criterion(
            cls_preds, reg_preds, centerness_preds, batch_cls_t, batch_reg_t, batch_center_t, batch_pos)
        loss = cls_loss + reg_loss + centerness_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_cls += cls_loss.item()
        sum_reg += reg_loss.item()
        sum_center += centerness_loss.item()

    n_batches = len(train_loader)
    print('epoch: {}, cls: {:.4f}, reg: {:.4f}, centerness: {:.4f}, elapsed time: {:.1f}'.format(
        epoch, sum_cls / n_batches, sum_reg / n_batches, sum_center / n_batches, time() - train_start))

## 推論（デコード・NMS）
学習したモデルで推論を行うための関数を定義する．分類スコア（Sigmoid）に**centernessスコア（Sigmoid）を掛け合わせる**ことで，物体の中心から離れた位置による低品質な予測（大きくずれたボックスになりやすい）のスコアを下げてからランキングする．これがcenternessブランチの役割であり，Anchorのように「不適切な予測を自然にフィルタする」仕組みを持たないFCOSにおいて，NMS前の誤検出抑制の鍵となる．クラスごとに閾値を超えたスコアの位置についてボックスをデコードし，`torchvision.ops.nms`で重複した検出結果を除去する．

In [ ]:
def predict(model, image_tensor, score_threshold=0.3, nms_threshold=0.5):
    model.eval()
    with torch.no_grad():
        cls_outputs, reg_outputs, centerness_outputs = model(image_tensor.unsqueeze(0).to(device))

    all_locations = torch.cat(LOCATIONS, dim=0)  # (total_locs, 2)
    cls_preds = torch.cat(cls_outputs, dim=1)[0]  # (total_locs, n_class - 1)
    reg_preds = torch.cat(reg_outputs, dim=1)[0]  # (total_locs, 4)
    centerness_preds = torch.cat(centerness_outputs, dim=1)[0]  # (total_locs,)

    # 分類スコアにcenternessスコアを掛け合わせ，中心から遠い位置の予測を減衰させる
    scores = cls_preds.sigmoid() * centerness_preds.sigmoid().unsqueeze(1)

    boxes_xyxy = torch.stack([
        all_locations[:, 0] - reg_preds[:, 0],
        all_locations[:, 1] - reg_preds[:, 1],
        all_locations[:, 0] + reg_preds[:, 2],
        all_locations[:, 1] + reg_preds[:, 3],
    ], dim=1).clamp(min=0, max=IMG_SIZE)

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(cls_preds.size(1)):
        class_scores = scores[:, class_idx]
        mask = class_scores > score_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_xyxy[mask]
        class_scores_masked = class_scores[mask]

        keep = nms(class_boxes, class_scores_masked, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores_masked[keep])
        result_labels.append(torch.full((len(keep),), class_idx + 1, dtype=torch.long))  # +1: VOC_CLASSESのラベルは1始まり

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用いる．mAPの算出には`ssd.ipynb`と同様に`torchmetrics`の`MeanAveragePrecision`（`iou_type='bbox'`）を利用し，VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認する．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画する．

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルFCOSとの違い
本ノートブックで実装したFCOSは，原論文（Tian et al., "FCOS: Fully Convolutional One-Stage Object Detection", ICCV 2019）の構成を一部簡略化・変更している．主な違いは次の通りである．

| | 原論文（FCOS） | 本ノートブック |
|---|---|---|
| バックボーン | ResNet50-FPN（ImageNet事前学習済み） | ResNet50-FPN（ImageNet事前学習済み，変更なし） |
| 入力画像サイズ | 短辺800/長辺1333（マルチスケール学習・テスト） | 512×512（固定・簡略化） |
| ヘッドの正規化層 | 各畳み込みの後にGroupNormを使用 | 正規化層なし（実装の簡略化のため） |
| Positive sampleの判定 | 改良版ではcenter sampling（GT中心付近の限定領域のみを正例にする）を追加 | 素朴な「ボックス内部 + サイズ範囲内」のみで判定（center samplingなし） |
| 回帰損失 | IoU LossまたはGIoU Loss | 要素ごとのIoU Loss（`box_iou`のようなペアワイズ計算ではなく自前実装） |
| NMS | 独自実装 | `torchvision.ops.nms`を使用 |

特徴ピラミッド（ResNet50-FPN，P3〜P7）・各位置への教師信号の割り当てルール（ボックス内部判定＋サイズ範囲によるレベル割り当て）・レベルごとの学習可能なスケール（`self.scales`）といったFCOSの核となる設計は原論文と同じであり，学習の安定化や実装量を抑えるための周辺部分（正規化層・center sampling・マルチスケール学習）を簡略化している点が主な変更点である．

## 課題

1. 推論時の`predict`関数で，`scores = cls_preds.sigmoid() * centerness_preds.sigmoid().unsqueeze(1)`からcenternessの重み付けを外し，`cls_preds.sigmoid()`のみでランキング・NMSを行った場合に，誤検出（物体の中心から離れた位置による低品質なボックス）がどのように増えるか確認しましょう．

2. `SIZE_RANGES`（各FPNレベルが担当する物体の大きさの範囲）を変更し，小さい物体・大きい物体の検出精度にどのような影響があるか確認しましょう．

3. 同じテスト画像に対して，`ssd.ipynb`（Anchorベース）と本ノートブック（Anchor-free）の検出結果を比較し，検出されたボックスの数・位置・スコアにどのような違いがあるか確認しましょう．